# 2 · DDI Atomic Triplet Extraction

In [1]:
import os
import pandas as pd
import spacy
from spacy.matcher import PhraseMatcher
from collections import defaultdict
import networkx as nx


## Load the biomedical parser

In [ ]:
pip install scispacy
pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

In [ ]:
def load_biomedical_parser():
    try:
        nlp = spacy.load("en_core_sci_sm")
        print("Loaded scispaCy en_core_sci_sm (biomedical parser).")
        return nlp
    except OSError:
        print(
            "[WARNING] en_core_sci_sm is not installed")
        return spacy.load("en_core_web_sm")


nlp = load_biomedical_parser()


[WARNING] en_core_sci_sm is not installed - falling back to generic en_core_web_sm. This does NOT match the methodology's specified scispaCy pipeline (§2.1.8/§3.3) and will likely produce noisier dependency parses on clinical text. Install it with:
    pip install scispacy
    pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz


## FDA drug vocabulary + entity-recognition helpers

In [4]:
df_raw_fda = pd.read_csv("../drug_products.csv", encoding="ISO-8859-1")

df_prescription = df_raw_fda[df_raw_fda["PRODUCTTYPENAME"] == "HUMAN PRESCRIPTION DRUG"]

fda_drug_names = set()
for col in ["PROPRIETARYNAME", "NONPROPRIETARYNAME", "SUBSTANCENAME"]:
    if col in df_prescription.columns:
        for val in df_prescription[col].dropna().unique():
            val_clean = str(val).lower().strip()
            for part in val_clean.split(";"):
                fda_drug_names.add(part.strip())

global_patterns = [nlp.make_doc(t) for t in fda_drug_names if len(t.strip()) > 2]
global_matcher  = PhraseMatcher(nlp.vocab, attr="LOWER")
global_matcher.add("FDA_NDC_INDEX", global_patterns)

print(f"Global drug vocabulary: {len(fda_drug_names):,} entries")


def check_entity_validity(text, e1, e2):
    """Deterministic OOV/hallucinated-entity check (fallback only - see markdown above)."""
    if not isinstance(text, str) or not text:
        return False
    t = text.lower()
    return (str(e1).lower().strip() in t) and (str(e2).lower().strip() in t)


Global drug vocabulary: 8,165 entries


## Extraction helpers

In [ ]:
DISCOURSE_VERBS = {
    "be", "say", "find", "read", "look", "talk", "go", "come",
    "know", "seem", "use", "make", "get", "have", "think", "check",
    "tell", "show", "mean", "note", "mention", "see", "hear",
    "explain", "describe", "share", "post", "write", "ask",
    "suggest", "indicate", "report", "demonstrate", "conduct", "study",
    "investigate", "examine", "evaluate", "assess", "publish", "document",
    "observe", "identify", "confirm", "discuss", "state", "highlight",
    "reveal", "consider", "review", "summarize", "outline",
}

SUBJECT_PRONOUNS = {"it", "its", "they", "their", "them", "this", "that"}

NOISE_DOBJS = {
    "stuff", "info", "information", "thing", "something", "study",
    "interaction", "data", "result", "report", "paper", "article",
    "drug", "medication", "med", "it", "they", "this", "that",
}


def build_row_matcher(e1: str, e2: str):
    row_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    row_matcher.add("FDA_NDC_INDEX", global_patterns)
    extra = set()
    for name in [e1, e2]:
        if name and len(str(name).strip()) > 2:
            extra.add(str(name).lower().strip())
    if extra:
        row_matcher.add("KNOWN_PAIR", [nlp.make_doc(n) for n in extra])
    return row_matcher, extra


def get_sdp(doc, tok_a, tok_b):
    edges = []
    for token in doc:
        for child in token.children:
            edges.append((token.i, child.i))
            edges.append((child.i, token.i))
    G = nx.Graph(edges)
    try:
        path_indices = nx.shortest_path(G, tok_a.i, tok_b.i)
        return [doc[i] for i in path_indices]
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return None


def sdp_to_relation(path_tokens):
    if len(path_tokens) < 3:
        return None
    middle = path_tokens[1:-1]
    parts = []
    for tok in middle:
        neg = any(c.dep_ == "neg" for c in tok.children)
        if tok.pos_ == "VERB":
            parts.append(("not " if neg else "") + tok.lemma_.lower())
        elif tok.pos_ in ("NOUN", "ADP", "PART"):
            parts.append(tok.text.lower())
        elif tok.dep_ == "neg":
            pass
    rel = " ".join(parts).strip()
    return rel if rel else None


def determine_direction(drug_a, drug_b):
    SUBJ_DEPS = {"nsubj", "nsubjpass", "csubj", "agent"}
    if drug_a.dep_ in SUBJ_DEPS:
        return drug_a, drug_b
    if drug_b.dep_ in SUBJ_DEPS:
        return drug_b, drug_a
    return (drug_a, drug_b) if drug_a.i < drug_b.i else (drug_b, drug_a)


def build_hypothesis(subj_text, relation, obj_text):
    return f"{subj_text.capitalize()} {relation} {obj_text.lower()}."


def extract_explicit(sent_doc, drug_tokens):
    """TIER 1: both drugs appear as matched tokens in this sentence."""
    triplets = []
    seen_pairs = set()
    for i, ta in enumerate(drug_tokens):
        for tb in drug_tokens[i + 1:]:
            if ta.text.lower() == tb.text.lower():
                continue
            pair_key = tuple(sorted([ta.i, tb.i]))
            if pair_key in seen_pairs:
                continue
            seen_pairs.add(pair_key)
            path = get_sdp(sent_doc, ta, tb)
            if path is None:
                continue
            relation = sdp_to_relation(path)
            if not relation:
                continue
            subj_tok, obj_tok = determine_direction(ta, tb)
            subj_name, obj_name = subj_tok.text.lower(), obj_tok.text.lower()
            hyp = build_hypothesis(subj_name, relation, obj_name)
            triplets.append((subj_name, relation, obj_name, hyp, "tier1_sdp"))
    return triplets


def extract_pronoun_coreference(sent_doc, drug_tokens, known_pair):
    """TIER 3: exactly 1 drug found; pronoun subject resolved to the partner drug."""
    triplets = []
    if len(drug_tokens) != 1:
        return triplets

    found_drug = drug_tokens[0].text.lower()
    partner_drugs = [d for d in sorted(known_pair) if d != found_drug]
    if not partner_drugs:
        return triplets
    partner = partner_drugs[0]

    drug_tok = drug_tokens[0]
    gov, current, visited = None, drug_tok.head, set()
    while current.i not in visited:
        visited.add(current.i)
        if current.pos_ in ("VERB", "AUX"):
            gov = current
            break
        if current.head.i == current.i:
            break
        current = current.head

    if gov is None or gov.lemma_.lower() in DISCOURSE_VERBS:
        return triplets

    pronoun_subj = None
    for child in gov.children:
        if child.dep_ in ("nsubj", "nsubjpass") and child.text.lower() in SUBJECT_PRONOUNS:
            pronoun_subj = child
            break
    if pronoun_subj is None:
        return triplets

    neg = any(c.dep_ == "neg" for c in gov.children)
    rel_parts = [("not " if neg else "") + gov.lemma_.lower()]
    for child in gov.children:
        if child.dep_ == "dobj" and child.pos_ == "NOUN":
            rel_parts.append(child.lemma_.lower())
        if child.dep_ == "prep":
            rel_parts.append(child.text.lower())

    relation = " ".join(rel_parts)
    hyp = build_hypothesis(partner, relation, found_drug)
    triplets.append((partner, relation, found_drug, hyp, "tier3_pronoun"))
    return triplets


def extract_coordinated_governor(sent_doc, drug_tokens):
    triplets = []
    seen_pairs = set()
    for i, ta in enumerate(drug_tokens):
        for tb in drug_tokens[i + 1:]:
            if ta.text.lower() == tb.text.lower():
                continue
            pair_key = tuple(sorted([ta.i, tb.i]))
            if pair_key in seen_pairs:
                continue
            path = get_sdp(sent_doc, ta, tb)
            if path is not None and len(path) > 2:
                continue   # Tier 1 already handles real multi-hop paths
            seen_pairs.add(pair_key)

            gov, current, visited, hops = None, ta, set(), 0
            while current.i not in visited and hops < 6:
                visited.add(current.i)
                if current.pos_ in ("VERB", "AUX") and current.lemma_.lower() not in DISCOURSE_VERBS:
                    gov = current
                    break
                if current.head.i == current.i:
                    break
                current = current.head
                hops += 1
            if gov is None:
                continue

            neg = any(c.dep_ == "neg" for c in gov.children)
            rel_parts = [("not " if neg else "") + gov.lemma_.lower()]
            for child in gov.children:
                if child.dep_ in ("dobj", "attr") and child.pos_ in ("NOUN", "ADJ"):
                    rel_parts.append(child.lemma_.lower())
                if child.dep_ == "prep" and child.text.lower() not in ("between", "of"):
                    rel_parts.append(child.text.lower())
            relation = " ".join(rel_parts).strip()
            if not relation:
                continue

            hyp = build_hypothesis(ta.text.lower(), relation, tb.text.lower())
            triplets.append((ta.text.lower(), relation, tb.text.lower(), hyp, "tier2_coordinated_governor"))
    return triplets


print("Extraction tiers defined.")


Extraction tiers defined.


## atomize_dataframe

In [6]:
def atomize_dataframe(
    df, *, id_col, text_col, premise_col="premise",
    e1_col="e1_text", e2_col="e2_text",
    label_col="label", scenario_col="scenario",
    entity_valid_col="entity_valid",
):
    """Decomposes every row's `text_col` into SVO atomic triplets.

    Returns a DataFrame with one row per extracted triplet, PLUS one
    'filtered_out' placeholder row for any source row that yielded zero
    triplets (so response-level coverage can be audited before those rows
    are dropped for training/evaluation).
    """
    atomic_rows = []

    for _, row in df.iterrows():
        text    = row[text_col]
        premise = row[premise_col]
        e1      = str(row[e1_col]).strip() if pd.notna(row[e1_col]) else ""
        e2      = str(row[e2_col]).strip() if pd.notna(row[e2_col]) else ""
        label    = row[label_col]
        scenario = row[scenario_col] if scenario_col in row else label
        entity_valid = row[entity_valid_col] if entity_valid_col in row else check_entity_validity(text, e1, e2)

        if pd.isna(text) or not str(text).strip():
            continue
        if e1.lower() == e2.lower():
            continue

        row_matcher, known_pair = build_row_matcher(e1, e2)
        doc = nlp(str(text))
        found_triplets = False

        for sentence in doc.sents:
            sent_doc = sentence.as_doc()   # FIX: reuse the parsed span, don't re-parse raw text

            matches = row_matcher(sent_doc)
            seen_starts = {}
            for match_id, start, end in matches:
                if start not in seen_starts:
                    seen_starts[start] = sent_doc[start]

            seen_names = {}
            for tok in seen_starts.values():
                seen_names.setdefault(tok.text.lower(), tok)
            drug_tokens = list(seen_names.values())

            if len(drug_tokens) >= 2:
                triplets = extract_explicit(sent_doc, drug_tokens)
                if not triplets:
                    # Tier 1 found the drugs but no real path between them
                    # (the "X and Y" coordination case) - try Tier 2 before
                    # giving up on this sentence.
                    triplets = extract_coordinated_governor(sent_doc, drug_tokens)
            elif len(drug_tokens) == 1:
                triplets = extract_pronoun_coreference(sent_doc, drug_tokens, known_pair)
            else:
                triplets = []

            for subj, rel, obj, hyp, tier in triplets:
                if not rel.strip():
                    continue
                found_triplets = True
                atomic_rows.append({
                    "original_id":  row[id_col],
                    "Entity1":      e1,
                    "Entity2":      e2,
                    "label":        label,          # FIX: was missing entirely - inherited from parent row
                    "scenario":     scenario,
                    "entity_valid": entity_valid,
                    "premise":      premise,
                    "source_text":  text,
                    "sub_extract":  subj,
                    "obj_extract":  obj,
                    "rel_extract":  rel,
                    "hypothesis":   hyp,             # FIX: renamed from hypothesis_text for MAIN_NB compatibility
                    "source_tier":  tier,
                })

        if not found_triplets:
            atomic_rows.append({
                "original_id":  row[id_col],
                "Entity1":      e1,
                "Entity2":      e2,
                "label":        label,
                "scenario":     scenario,
                "entity_valid": entity_valid,
                "premise":      premise,
                "source_text":  text,
                "sub_extract":  None,
                "obj_extract":  None,
                "rel_extract":  None,
                "hypothesis":   None,
                "source_tier":  "filtered_out",
            })

    return pd.DataFrame(atomic_rows)


def report_coverage(df_atomic, n_source_rows, split_name):
    covered = df_atomic.loc[df_atomic["source_tier"] != "filtered_out", "original_id"].nunique()
    print(f"[{split_name}] coverage: {covered}/{n_source_rows} source rows "
          f"({100*covered/max(n_source_rows,1):.1f}%) produced >=1 triplet")
    if "scenario" in df_atomic.columns:
        by_scn = df_atomic.groupby("scenario")["source_tier"].apply(
            lambda s: (s != "filtered_out").mean()
        )
        print("  coverage by scenario:")
        print("  " + by_scn.round(3).to_string().replace("\n", "\n  "))
    print(f"  tier breakdown:\n  " + df_atomic["source_tier"].value_counts().to_string().replace("\n", "\n  "))
    print()


print("atomize_dataframe() and report_coverage() defined.")


atomize_dataframe() and report_coverage() defined.


## Run atomisation

In [7]:
holistic_train = pd.read_csv("data/holistic_train.csv")
holistic_val   = pd.read_csv("data/holistic_val.csv")
holistic_test  = pd.read_csv("data/holistic_test.csv")

print(f"holistic_train: {len(holistic_train):,}  holistic_val: {len(holistic_val):,}  holistic_test: {len(holistic_test):,}")


holistic_train: 23,028  holistic_val: 4,064  holistic_test: 4,780


In [8]:
atomic_train_raw = atomize_dataframe(holistic_train, id_col="original_id", text_col="hypothesis")
report_coverage(atomic_train_raw, len(holistic_train), "atomic_train")

atomic_val_raw = atomize_dataframe(holistic_val, id_col="original_id", text_col="hypothesis")
report_coverage(atomic_val_raw, len(holistic_val), "atomic_val")

atomic_test_raw = atomize_dataframe(holistic_test, id_col="original_id", text_col="hypothesis")
report_coverage(atomic_test_raw, len(holistic_test), "atomic_test")


[atomic_train] coverage: 19453/23028 source rows (84.5%) produced >=1 triplet
  coverage by scenario:
  scenario
  contradiction    0.880
  entailment       0.904
  neutral          0.818
  tier breakdown:
  source_tier
  tier1_sdp                     10032
  tier2_coordinated_governor     9794
  filtered_out                   3148
  tier3_pronoun                     2

[atomic_val] coverage: 3405/4064 source rows (83.8%) produced >=1 triplet
  coverage by scenario:
  scenario
  contradiction    0.865
  entailment       0.890
  neutral          0.831
  tier breakdown:
  source_tier
  tier1_sdp                     1786
  tier2_coordinated_governor    1685
  filtered_out                   569

[atomic_test] coverage: 3368/4780 source rows (70.5%) produced >=1 triplet
  coverage by scenario:
  scenario
  contradiction    0.978
  entailment       0.954
  fake_drug        0.948
  neutral          0.671
  tier breakdown:
  source_tier
  tier1_sdp                     13320
  filtered_out     

## Drop f iltered out


In [9]:
atomic_train = atomic_train_raw[atomic_train_raw["source_tier"] != "filtered_out"].reset_index(drop=True)
atomic_val   = atomic_val_raw[atomic_val_raw["source_tier"] != "filtered_out"].reset_index(drop=True)
atomic_test  = atomic_test_raw[atomic_test_raw["source_tier"] != "filtered_out"].reset_index(drop=True)

os.makedirs("data", exist_ok=True)
atomic_train.to_csv("data/atomic_train.csv", index=False)
atomic_val.to_csv("data/atomic_val.csv", index=False)
atomic_test.to_csv("data/atomic_test.csv", index=False)

print(f"Saved data/atomic_train.csv  - {len(atomic_train):,} triplets")
print(f"Saved data/atomic_val.csv    - {len(atomic_val):,} triplets")
print(f"Saved data/atomic_test.csv   - {len(atomic_test):,} triplets")

print("\nLabel distribution — atomic_train:")
print(atomic_train["label"].value_counts())
print("\nLabel distribution — atomic_val:")
print(atomic_val["label"].value_counts())
print("\nLabel distribution — atomic_test:")
print(atomic_test["label"].value_counts())


Saved data/atomic_train.csv  - 19,828 triplets
Saved data/atomic_val.csv    - 3,471 triplets
Saved data/atomic_test.csv   - 14,025 triplets

Label distribution — atomic_train:
label
neutral          7388
entailment       6389
contradiction    6051
Name: count, dtype: int64

Label distribution — atomic_val:
label
neutral          1314
entailment       1109
contradiction    1048
Name: count, dtype: int64

Label distribution — atomic_test:
label
neutral          6434
entailment       4083
contradiction    3508
Name: count, dtype: int64


## Check — sample triplets

In [10]:
print("Relation distribution (train):")
print(atomic_train["rel_extract"].value_counts().head(20).to_string())
print()
print("Sample triplets:")
atomic_test[["Entity1", "Entity2", "sub_extract", "rel_extract", "obj_extract", "source_tier", "entity_valid"]].head(20)


Relation distribution (train):
rel_extract
lead to                     387
agents                      343
combine                     307
utilize                     220
of use                      211
regard                      207
drugs                       207
utilize in                  198
antidepressants             196
inhibitors                  185
expect shift                184
not support interference    183
follow                      175
involve                     172
appear during               169
remain                      168
sodium                      165
of administration           165
between interaction         165
reference in                164

Sample triplets:


,Entity1,Entity2,sub_extract,rel_extract,obj_extract,source_tier,entity_valid
0,abacavir,lamivudine,lamivudine,of addition not alter properties of,abacavir,tier1_sdp,True
1,abacavir,lamivudine,zidovudine,lamivudine of addition not alter properties of,abacavir,tier1_sdp,True
2,abacavir,lamivudine,abacavir,exposure remain use with,lamivudine,tier1_sdp,True
3,abacavir,lamivudine,abacavir,exposure remain use with lamivudine,zidovudine,tier1_sdp,True
4,abacavir,lamivudine,abacavir,of coadministration,lamivudine,tier1_sdp,True
5,abacavir,lamivudine,lamivudine,combine with agents as,zidovudine,tier1_sdp,True
6,abacavir,lamivudine,lamivudine,combine observe give with,abacavir,tier1_sdp,True
7,abacavir,lamivudine,zidovudine,as agents with combine observe give with,abacavir,tier1_sdp,True
8,abacavir,lamivudine,abacavir,design,lamivudine,tier2_coordinated_governor,True
9,abacavir,lamivudine,abacavir,of properties not alter by addition of,zidovudine,tier1_sdp,False
